# Example Notebook: Stock market returns and ruling party

This notebook shows how we can translate a research question into a testable hypothesis, find data and test it empirically.

### 1. Research Question:
Does the stock market perform better when Republicans are in office?


### 2. Testable hypothesis:
When the US has a Republican president, real stock market returns tend to be higher than when a Democrat is in office.

In [ ]:
# Load necessary libraries
import statsmodels.api as sm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import datetime
%pip install fredapi
import fredapi

### 3. Finding Data using FRED
We need to find data on **real stock market returns** and an indicator of the **president's party affiliation**.

In [ ]:
# Set up FRED data connection
# Instructions for getting your FRED API key available at these websites
# https://pypi.org/project/fredapi/ 
# http://api.stlouisfed.org/api_key.html
from fredapi import Fred
# The key is read from the FRED_API_KEY environment variable; never paste it here.
fred = Fred()

# OECD Total share price index 
temp = fred.get_series('SPASTT01USM661N', observation_start='1960-01-01', observation_end='2024-07-01') 
df_total_share_price_idx = pd.DataFrame(data = temp, columns=['Total_Price_Index']) 
df_total_share_price_idx = df_total_share_price_idx.rename_axis('Date').reset_index() 
print(df_total_share_price_idx.head(5))

# Annual dividends
temp = fred.get_series('A2218C1A027NBEA', observation_start='1929-01-01', observation_end='2021-01-01')
df_dividends = pd.DataFrame(data = temp, columns=['Total_Dividends_Paid'])
df_dividends = df_dividends.rename_axis('Date').reset_index()
print(df_dividends.head(5))

# Dividends are measured in total annual dollars. Convert to index, base 2015 to align with share price data
div_base = np.float64(df_dividends.loc[df_dividends["Date"] == "2015-01-01", "Total_Dividends_Paid"])
df_dividends['Total_Dividends_Index'] =  ((df_dividends['Total_Dividends_Paid']/12)/div_base)*100
print(df_dividends.head(5))

# Load CPI 
temp = fred.get_series('CPIAUCSL', observation_start='1960-01-01', observation_end='2023-07-01')
df_cpi = pd.DataFrame(data = temp, columns=['CPI'])
df_cpi = df_cpi.rename_axis('Date').reset_index()

cpi_base = np.float64(df_cpi.loc[df_cpi["Date"] == "2023-07-01", "CPI"])
df_cpi['CPI_ADJ'] = cpi_base / df_cpi['CPI']
print(df_cpi.head(5))

In [ ]:
# Combine FRED data and define real return for each month
df_fred = pd.merge(df_total_share_price_idx, df_dividends, left_on='Date', right_on='Date')
df_fred = pd.merge(df_fred, df_cpi, left_on='Date', right_on='Date')

df_fred['Real_Price_Index'] = df_fred['Total_Price_Index']*df_fred['CPI_ADJ']
df_fred['Real_Dividends_Index'] = df_fred['Total_Dividends_Index']*df_fred['CPI_ADJ']
df_fred['Real_Price_Index_Lag'] = df_fred['Real_Price_Index'].shift()

df_fred['Real_Month_Return'] = (df_fred['Real_Price_Index'] + df_fred['Real_Dividends_Index']) / df_fred['Real_Price_Index_Lag']
df_fred['Real_Total_Return'] = df_fred['Real_Price_Index']
df_fred['Real_Month_Return'].where(np.arange(len(df_fred))>0, 0).add(1).cumprod() * df_fred.loc[0, 'Real_Total_Return']
df_fred['log_Real_Total_Return'] = np.log(df_fred['Real_Total_Return'])
df_fred['log_Real_Month_Return'] = np.log(df_fred['Real_Month_Return'])
print(df_fred)

In [ ]:
# Read in data files 
# GOP term flags
url = "../presidential-term-data/GOP_flag.csv"  # in this repository
temp = pd.read_csv(url,dtype=str)
df_gop = pd.DataFrame(temp)

# The file stores each month as a decimal number, YYYY.MM, so October lost its trailing zero:
# it is written 1960.1, not 1960.10, and reading it as a date makes it a second January.
# Split year from month and pad the month on the RIGHT: '1' becomes '10'; '01'...'12' are unchanged.
parts = df_gop['Date'].str.split('.', expand=True)
month = parts[1].str.ljust(2, '0')
df_gop['Date'] = pd.to_datetime(parts[0] + '-' + month, format='%Y-%m')  # first day of the month

# Check: one row per month, with none repeated and none missing
assert df_gop['Date'].is_unique
assert len(df_gop) == len(pd.date_range(df_gop['Date'].min(), df_gop['Date'].max(), freq='MS'))
print(df_gop.head())

In [ ]:
# Create new data frame that merges GOP flag with returns data
df = pd.merge(df_fred, df_gop, left_on='Date', right_on='Date')
print(df.head(5))

In [ ]:
# Plot real total returns. Shade GOP presidential terms
# convenience function
def plot_series(ax, df, index='Date', cols=['Real_Total_Return'], area='GOP'):
    # convert area variable to boolean
    df[area] = df[area].astype(int).astype(bool)
    # set up an index based on date
    df = df.set_index(keys=index, drop=False)
    # line plot
    df.plot(ax=ax, x=index, y=cols, color='blue')
    # extract limits
    y1, y2 = ax.get_ylim()
    ax.fill_between(df[index].index, y1=y1, y2=y2, where=df[area], facecolor='grey', alpha=0.4)
    return ax

# set up figure, axis
f, ax = plt.subplots()
plot_series(ax, df)
ax.grid(True)
plt.show()


In [ ]:
# Plot log total returns over time. Shade GOP presidential terms
# convenience function
def plot_series(ax, df, index='Date', cols=['log_Real_Month_Return'], area='GOP'):
    # convert area variable to boolean
    df[area] = df[area].astype(int).astype(bool)
    # set up an index based on date
    df = df.set_index(keys=index, drop=False)
    # line plot
    df.plot(ax=ax, x=index, y=cols, color='blue')
    # extract limits
    y1, y2 = ax.get_ylim()
    ax.fill_between(df[index].index, y1=y1, y2=y2, where=df[area], facecolor='grey', alpha=0.4)
    return ax

# set up figure, axis
f, ax = plt.subplots()
plot_series(ax, df)
ax.grid(True)
plt.show()


### 4. Testing hypothesis empirically with OLS

In [ ]:
# Run OLS regression of Monthly earnings onto GOP president flag
# Outcome real earnings next period. GOP president is pre-determined. 
df['GOPlag'] = df['GOP'].shift()

results = sm.formula.ols(formula = 'log_Real_Month_Return ~ GOPlag', data=df).fit(cov_type='HC3')
print(results.summary())

### 5. Conclusion
The regression analysis shows that the stock market performs worse under a GOP presidency.